# PPO с Continuous Action Space (TensorBoard + Weights & Biases)

Этот ноутбук демонстрирует обучение PPO (Stable Baselines 3) на среде `TradingEnvV5`. В качестве сети используется кастомный `GatedMlpExtractor` (FiLM), который маршрутизирует вероятности от HMM-модели, выступая в роли вентиля для фичей рынка.

In [11]:
%load_ext autoreload
%autoreload 2
%load_ext tensorboard

import os
import wandb
import pandas as pd
import numpy as np
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from wandb.integration.sb3 import WandbCallback

from core.data.data_loader import load_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5
from agents.ppo_agent import create_ppo_agent, TradingMetricsCallback

# ==========================================
# НАСТРОЙКИ МОДЕЛИ И ДАННЫХ
# ==========================================
# MODEL_PATH = "models/ppo_gated_agent_final"         # <--- Укажите путь к модели
# NORM_PATH = "models/vec_normalize_final.pkl"  # <--- Укажите путь к нормализатору

# Инициализация Weights & Biases
# sync_tensorboard=True заставит W&B автоматически копировать все графики из TensorBoard в облако
# run = wandb.init(
#     project="trading_rl",
#     name=exp_widget.value,
#     sync_tensorboard=True,  
#     monitor_gym=True,       
#     save_code=True,
# )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:
from core.experiment_manager import get_train_widgets, get_experiment_paths
exp_widget = get_train_widgets(default_name="ppo_gated")

# Автоматическое получение путей
model_path, norm_path, tensorboard_log = get_experiment_paths(exp_widget.value)

# Поддержка старых переменных в ноутбуке
model_widget = type('obj', (object,), {'value': model_path})
norm_widget = type('obj', (object,), {'value': norm_path})
MODEL_PATH = model_path
NORM_PATH = norm_path
vec_normalize_path = norm_path


## 1. Загрузка данных и генерация фичей (Включая HMM)

In [5]:
df = load_crypto_data(symbol='BTCUSDT', start_date='2024-01-01', interval='4h', source='bybit_futures')

fg = FeatureGenerator()
data_features = fg.transform(df)
print(f"Data Features shape: {data_features.shape}")

# Проверяем, что вероятности HMM добавились
hmm_cols = [c for c in data_features.columns if 'hmm_regime' in c]
print(f"HMM Columns: {hmm_cols}")

✅ All data in cache, loading from disk...
✅ HMM Model loaded from /Users/maximsinyaev/projects/trading_rl/trading_rl/models/hmm_model.pkl
Data Features shape: (5336, 48)
HMM Columns: ['hmm_regime_0_prob', 'hmm_regime_1_prob', 'hmm_regime_2_prob']


## 2. Создание векторизованной среды и нормализация

In [6]:
def make_env():
    env = TradingEnvV5(df=data_features, continuous_action=True, t_max=1000)
    # Оборачиваем в Monitor для записи эпизодов (rewards, lengths) в TensorBoard
    return Monitor(env)

# В SB3 PPO ожидает векторизованную среду
vec_env = DummyVecEnv([make_env])

# Нормализуем наблюдения и награды. ОЧЕНЬ ВАЖНО для PPO.
vec_env = VecNormalize(
    vec_env,
    norm_obs=True,
    norm_reward=True,
    clip_obs=10.,
    clip_reward=10.
)

## 3. Запуск TensorBoard

Локальный сервер для графиков (можно также смотреть всё в дашборде W&B).

In [ ]:
%tensorboard --logdir ./tensorboard_logs/

## 4. Создание PPO Агента (с Gated-сетью)

In [8]:
# num_hmm_states берем из количества колонок, созданных FeatureGenerator
num_hmm_states = len(hmm_cols)

# use_gating=True активирует GatedMlpExtractor
model = create_ppo_agent(vec_env,  num_hmm_states=num_hmm_states, tensorboard_log=tensorboard_log)

🚀 Initializing PPO (MLP) on device: mps


/Users/maximsinyaev/projects/trading_rl/trading_rl/.venv/lib/python3.12/site-packages/stable_baselines3/common/policies.py:486: UserWarning:

As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])

/Users/maximsinyaev/projects/trading_rl/trading_rl/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning:

You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.



## 5. Обучение

In [9]:
# WandbCallback автоматически сохраняет лучшую модель и логгирует дополнительные вещи.
# TradingMetricsCallback отправляет PnL в TensorBoard (а W&B это перехватывает).
callbacks = [
    TradingMetricsCallback(),
    # WandbCallback(
    #     gradient_save_freq=1000,
    #     model_save_path=f"models/{run.id}",
    #     verbose=0,
    # )
]

print("Начинаем обучение PPO...")
# 500_000 шагов для хорошего бейзлайна
model.learn(total_timesteps=50_000, callback=callbacks, progress_bar=True)

# Сохраняем итоговую модель и нормализатор
model.save(MODEL_PATH)
vec_env.save(NORM_PATH)
print("Модель сохранена!")
wandb.finish()

Output()

Начинаем обучение PPO...


Модель сохранена!


env/mean_pnl,▁▁▂▁▁▂▂▁▂▃▂▄▂▂▃▄▃▄▄▄▇▅▃█▅
env/mean_real_reward,▁▁▂▂▄▃▃▁▁▅▂▂▁▁▄▅▅▄▄▄▁▁▁▁▇▅▅▂▂▃▄▄▄██▄▆▆▆▇
global_step,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇█████
rollout/ep_len_mean,▂▁▄▄▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇██
rollout/ep_rew_mean,▇█▅▅▅▅▄▅▅▅▅▄▄▄▃▃▂▃▂▂▂▂▁▁▁
time/fps,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/approx_kl,▁▂▂▂▂▂▃▃▃▃▃▄▅▄▄▄▅▅▅█▅█▆▆
train/clip_fraction,▁▂▂▃▃▄▅▅▅▅▆▆▆▆▆▆▆▇▇█▇▇▇▇
train/clip_range,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/entropy_loss,▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▅▆▆▇▇▇▇██
+6,...


## 🔬 Универсальный Блок Валидации
Этот блок можно запускать отдельно в любой момент. Он загружает обученную модель с диска, прогоняет её по тестовому куску данных и рисует красивый график цены, совмещенный с кривой вашего депозита.


In [19]:

from core.evaluation.validation import run_validation

# Запускаем симуляцию случайного отрезка графика
run_validation(
    data_features=data_features,
    model_path=MODEL_PATH,
    norm_path=NORM_PATH,
    use_frame_stack=False,
    n_stack=10,
    test_size=360,
    random_start=True
)


Loading model from models/ppo_gated_agent_final...
Random validation slice selected: from index 4113 to 4473.
Running simulation...

📊 VALIDATION REPORT
Initial balance: $100,000.00
Final balance:   $95,472.10
Net Profit:      -4.53%
Candles in pos:  337 out of 359
Status: 🏁 Successfully reached end of slice
